# MinerU `text_level` Refinement Agent

**Purpose**: Correct `text_level` tags in MinerU's `_content_list.json` output using Gemini 2.5 Flash VLM-based TOC extraction.

**Pipeline**:
1. **VLM TOC Extraction** — Send PDF to Gemini 2.5 Flash → extract section headings as structured JSON
2. **Human-in-the-Loop Review** — Display & confirm the LLM's TOC detection result before proceeding
3. **Fuzzy Match & Correct** — Use confirmed ground truth to fix `text_level` in `content_list.json`

**Dependencies**: `google-genai`, `rapidfuzz`, `PyMuPDF (fitz)`

In [1]:
import json
import os
import re
from pathlib import Path

from IPython.display import display, Markdown
from google import genai
from google.genai import types
from rapidfuzz import fuzz

# ============================================================
# CONFIGURATION — Edit these variables before running
# ============================================================
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
GEMINI_MODEL = 'gemini-2.5-flash'

DOC_CODE = "g33a"
PDF_PATH  = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\{DOC_CODE}\auto\{DOC_CODE}_origin.pdf"
JSON_PATH = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\{DOC_CODE}\auto\{DOC_CODE}_content_list copy.json"

OUTPUT_DIR = None  # Set to a directory path, or None to save alongside input JSON

# Human-in-the-loop temp file
HITL_JSON_PATH = None  # Auto-generated from JSON_PATH if None

print('Configuration loaded')
print(f'  Model: {GEMINI_MODEL}')
print(f'  PDF:   {PDF_PATH}')
print(f'  JSON:  {JSON_PATH}')

Configuration loaded
  Model: gemini-2.5-flash
  PDF:   C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\g33a\auto\g33a_origin.pdf
  JSON:  C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\g33a\auto\g33a_content_list copy.json


In [2]:
# ============================================================
# Load content_list.json and show statistics
# ============================================================
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    content_list = json.load(f)

total_blocks = len(content_list)
text_level_blocks = [b for b in content_list if 'text_level' in b]
pages = set(b.get('page_idx', -1) for b in content_list)

print(f'Content List Statistics')
print(f'  Total blocks:          {total_blocks}')
print(f'  Blocks with text_level: {len(text_level_blocks)}')
print(f'  Page range:            {min(pages)} - {max(pages)}')
print(f'\n--- Current text_level blocks ---')
for i, b in enumerate(text_level_blocks):
    print(f'  [{i+1:3d}] page {b["page_idx"]:3d} | {b.get("text", "")[:80]}')

Content List Statistics
  Total blocks:          310
  Blocks with text_level: 27
  Page range:            0 - 41

--- Current text_level blocks ---
  [  1] page   2 | 1. Introduction 
  [  2] page   3 | 2. Customer acceptance policy 
  [  3] page   3 | 3. Customer due diligence 
  [  4] page   6 | 4. Corporate customers 
  [  5] page   7 | 5. Trust and nominee accounts 
  [  6] page   8 | 6. Reliance on intermediaries for customer due diligence 
  [  7] page  10 | 7. Client accounts 
  [  8] page  10 | 8. Non-face-to-face customers 
  [  9] page  11 | 9. Wire transfer messages 
  [ 10] page  12 | 10. Politically exposed persons 
  [ 11] page  14 | 11. Correspondent banking 
  [ 12] page  15 | 12. Existing accounts 
  [ 13] page  15 | 13. On-going monitoring 
  [ 14] page  16 | 14. Jurisdictions which do not or insufficiently apply the FATF Recommendations 
  [ 15] page  18 | 15. Terrorist financing 
  [ 16] page  20 | 16. Risk management 
  [ 17] page  24 | INTERPRETATIVE NOTES 
  [ 1

## Stage 1: VLM TOC Extraction

Send the PDF document to **Gemini 2.5 Flash** to extract:
1. Which pages contain the Table of Contents (`toc_pages`, 0-indexed)
2. All **top-level** section headings listed in the TOC (`sections`)

The model receives the native PDF bytes and returns structured JSON.

In [3]:
# ============================================================
# Stage 1: VLM TOC Extraction using Gemini 2.5 Flash
# ============================================================

# Initialize Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

# Read PDF bytes
with open(PDF_PATH, 'rb') as f:
    pdf_bytes = f.read()
print(f'PDF loaded: {len(pdf_bytes):,} bytes')

# System prompt for TOC extraction
SYSTEM_PROMPT = """
You are a Document Structure Analyst. Your task is to identify and extract the primary structural hierarchy from a document's Table of Contents (TOC).

TASK:
1. Identify the indexed page numbers containing the TOC.
2. Extract the TOP-LEVEL sections only. A top-level section is defined as:
   - Numerical headers (e.g., "1", "2")
   - Formal Articles (e.g., "Article (1)", "Article 2")
   - Major Parts or Chapters (e.g., "PART I", "CHAPTER ONE")
   - Supplemental sections (e.g., "Annex A", "Appendix", "Schedule")

RULES:
- EXCLUDE sub-sections, nested bullet points, and indented descriptions (e.g., if "16 The Accuracy Obligation" is the section, do not include "Requirement of reasonable effort" listed under it).
- EXCLUDE page numbers, dot leaders (....), and extra whitespace.
- CLEAN the title text to remove trailing punctuation.
- "section_id": The numeric identifier (e.g., "1", "Article (1) to 1.", "Annex A"). If it is a PART title without a specific number, use null.
- "title": The full name of the section.

Return ONLY valid JSON:
{
  "sections": [
    {"section_id": "string or null", "title": "string"}
  ]
}"""

# Call Gemini 2.5 Flash
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
        'Extract the Table of Contents structure from this PDF document. Return the result as JSON.'
    ],
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        response_mime_type='application/json',
        temperature=0,
    )
)

# Parse result
raw_text = response.text
toc_result = json.loads(raw_text)

print(f'\nVLM TOC Extraction Complete')
print(f'  TOC pages (0-indexed): {toc_result.get("toc_pages", [])}')
print(f'  Sections found: {len(toc_result.get("sections", []))}')
print(f'\n--- Extracted Sections ---')
print(json.dumps(toc_result, indent=2))

PDF loaded: 212,472 bytes

VLM TOC Extraction Complete
  TOC pages (0-indexed): []
  Sections found: 18

--- Extracted Sections ---
{
  "sections": [
    {
      "section_id": "1",
      "title": "Introduction"
    },
    {
      "section_id": "2",
      "title": "Customer acceptance policy"
    },
    {
      "section_id": "3",
      "title": "Customer due diligence"
    },
    {
      "section_id": "4",
      "title": "Corporate customers"
    },
    {
      "section_id": "5",
      "title": "Trust and nominee accounts"
    },
    {
      "section_id": "6",
      "title": "Reliance on intermediaries for customer due diligence"
    },
    {
      "section_id": "7",
      "title": "Client accounts"
    },
    {
      "section_id": "8",
      "title": "Non-face-to-face customers"
    },
    {
      "section_id": "9",
      "title": "Wire transfer messages"
    },
    {
      "section_id": "10",
      "title": "Politically exposed persons"
    },
    {
      "section_id": "11",
      "ti

## Human-in-the-Loop Review

The LLM's TOC extraction result is saved to a JSON file for your review.

**How to review:**
1. Inspect the output above — check if all section headings are correct
2. If needed, open and edit the saved JSON file (path shown below)
3. Run the next cell and type `yes` to confirm, or `edit` if you modified the file

**What to look for:**
- Missing section headings
- Incorrect section IDs
- Wrong TOC page boundaries
- Titles with leftover dots or page numbers

In [4]:
# ============================================================
# Human-in-the-Loop: Review & Confirm TOC Result
# ============================================================

# Determine save path for the ground truth JSON
if HITL_JSON_PATH is None:
    json_dir = Path(JSON_PATH).parent
    HITL_JSON_PATH = str(json_dir / 'toc_ground_truth.json')

# Save LLM result to file for review / editing
with open(HITL_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(toc_result, f, indent=2, ensure_ascii=False)

print(f'Saved TOC result to: {HITL_JSON_PATH}')
print(f'\n{"="*70}')
print(f'LLM TOC Detection Result (review below)')
print(f'{"="*70}')
print(f'\nSections ({len(toc_result["sections"])} total):')
print(f'{"-"*70}')
for i, s in enumerate(toc_result['sections'], 1):
    sid = s.get('section_id') or 'PART'
    print(f' {sid}. {s["title"]}')
print(f'{"-"*70}')

# ---- Confirmation gate ----
print(f'\nREVIEW the result above.')
print(f'If you need to edit, open the file at:\n  {HITL_JSON_PATH}')
confirm = input('\nType "yes" to confirm, or "edit" if you edited the file: ').strip().lower()

if confirm in ('edit', 'e'):
    # Reload the user-edited file
    with open(HITL_JSON_PATH, 'r', encoding='utf-8') as f:
        toc_result = json.load(f)
    print(f'\nReloaded edited file. Sections: {len(toc_result["sections"])}')
    # Show updated result
    for i, s in enumerate(toc_result['sections'], 1):
        sid = s.get('section_id') or 'PART'
        print(f' {i:3d}. [{sid:>5}] {s["title"]}')
elif confirm in ('yes', 'y'):
    print('\nConfirmed! Proceeding with correction...')
else:
    raise RuntimeError(f'Aborted. Got "{confirm}". Re-run this cell after review.')

# Store confirmed ground truth for Stage 2
# toc_pages = set(toc_result.get('toc_pages', []))
ground_truth_sections = toc_result['sections']
print(f'\nGround Truth: {len(ground_truth_sections)} sections')    

Saved TOC result to: C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\g33a\auto\toc_ground_truth.json

LLM TOC Detection Result (review below)

Sections (18 total):
----------------------------------------------------------------------
 1. Introduction
 2. Customer acceptance policy
 3. Customer due diligence
 4. Corporate customers
 5. Trust and nominee accounts
 6. Reliance on intermediaries for customer due diligence
 7. Client accounts
 8. Non-face-to-face customers
 9. Wire transfer messages
 10. Politically exposed persons
 11. Correspondent banking
 12. Existing accounts
 13. On-going monitoring
 14. Jurisdictions which do not or insufficiently apply the FATF Recommendations
 15. Terrorist financing
 16. Risk management
 Annex. Intermediary certificate
 PART. Interpretative Notes
----------------------------------------------------------------------

REVIEW the result above.
If you need to edit, open the file at:
  C:\Users\User\De

## Stage 2: Fuzzy Match & Correct `content_list.json`

Using the confirmed ground truth, apply corrections:

1. **Remove TOC page tags** — All `text_level` on TOC pages are removed (those are TOC entries, not body headings)
2. **Remove false positives** — Body blocks with `text_level: 1` that don't match any ground truth section
3. **Add false negatives** — Body blocks that match ground truth but currently lack `text_level`

In [5]:
# ============================================================
# Stage 2: Fuzzy Matching & Correction Engine
# ============================================================

# Higher threshold since ground truth titles MUST exist in the body text
# MinerU errors are layout misclassification, not content errors
FUZZY_THRESHOLD = 95     # strict: titles should match very closely
TOKEN_SET_THRESHOLD = 92  # for token-set ratio pass (handles minor OCR gaps)
MIN_LEN_RATIO = 0.7       # block text length must be >= 70% of target length

def normalize(text):
    """Normalize text for comparison: lowercase, strip, collapse whitespace,
    remove trailing dots and page numbers."""
    # text = re.sub(r'[\.…]+\s*\d*\s*$', '', text)   # trailing dots + page nums
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text


def build_match_targets(sections):
    """Build list of (normalized_title, section_id, original_title) for matching."""
    targets = []
    for s in sections:
        title = s['title']
        sid = s.get('section_id')
        norm = normalize(title)
        targets.append((norm, sid, title))
        # Also build variant with section_id prefix e.g. "6 Organisations"
        if sid:
            with_id = normalize(f'{sid} {title}')
            targets.append((with_id, sid, title))
    return targets


def match_block(block_text, targets):
    """Check if a block's text matches any ground truth section.
    Returns (matched, section_title, score, method)."""
    norm_text = normalize(block_text)
    if not norm_text:
        return False, None, 0, None

    best_score = 0
    best_title = None

    # Pass 1: fuzz.ratio — strict character-level similarity
    for norm_target, sid, orig_title in targets:
        score = fuzz.ratio(norm_text, norm_target)
        if score > best_score:
            best_score = score
            best_title = orig_title
        if score >= FUZZY_THRESHOLD:
            return True, orig_title, score, 'ratio'

    # Pass 2: token_set_ratio — handles minor word reordering or OCR gaps
    # Higher threshold + strict length guard (block must not be much shorter than target)
    # for norm_target, sid, orig_title in targets:
    #     score = fuzz.token_set_ratio(norm_text, norm_target)
    #     if score >= TOKEN_SET_THRESHOLD:
    #         len_ratio = min(len(norm_text), len(norm_target)) / max(len(norm_text), len(norm_target))
    #         if len_ratio >= MIN_LEN_RATIO:
    #             return True, orig_title, score, 'token_set'

    return False, best_title, best_score, None  # best_score for debug


# Build match targets from confirmed ground truth
targets = build_match_targets(ground_truth_sections)

# Deep copy for correction
corrected = json.loads(json.dumps(content_list))
corrections = {'added': [], 'removed': [], 'kept': [], 'toc_removed': []}

for i, block in enumerate(corrected):
    page = block.get('page_idx', -1)
    page = int(page)+1
    has_level = 'text_level' in block
    block_text = block.get('text', '')
    block_type = block.get('type', '')

    # Skip non-text block types (image, table, page_number, header, etc.)
    if block_type not in ('text',):
        if has_level:
            del block['text_level']
            corrections['removed'].append({
                'index': i, 'page': page, 'text': block_text[:80],
                'reason': f'non-text block type: {block_type}'
            })
        continue

    # TOC page blocks: remove all text_level (TOC entries ≠ body headings)
    # if page in toc_pages:
    #     if has_level:
    #         del block['text_level']
    #         corrections['toc_removed'].append({
    #             'index': i, 'page': page, 'text': block_text[:80]
    #         })
    #     continue

    # Body page blocks: match against ground truth
    matched, match_title, score, method = match_block(block_text, targets)

    if has_level and matched:
        # Correct — keep text_level
        corrections['kept'].append({
            'index': i, 'page': page, 'text': block_text[:80],
            'matched': match_title, 'score': score, 'method': method
        })
    elif has_level and not matched:
        # False positive — MinerU incorrectly tagged this as heading
        del block['text_level']
        corrections['removed'].append({
            'index': i, 'page': page, 'text': block_text[:80],
            'reason': f'no ground truth match (best score: {score})'
        })
    elif not has_level and matched:
        # False negative — MinerU missed this heading
        block['text_level'] = 1
        corrections['added'].append({
            'index': i, 'page': page, 'text': block_text[:80],
            'matched': match_title, 'score': score, 'method': method
        })

# ---- Summary ----
print('Correction Complete')
print('=' * 70)
print(f'  Kept (correct):        {len(corrections["kept"])}')
print(f'  Removed (false pos):   {len(corrections["removed"])}')
print(f'  Removed (TOC pages):   {len(corrections["toc_removed"])}')
print(f'  Added (false neg):     {len(corrections["added"])}')
print('=' * 70)

if corrections['removed']:
    print('\n--- Removed (False Positives) ---')
    for c in corrections['removed']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]} | reason: {c["reason"]}')

if corrections['toc_removed']:
    print('\n--- Removed (TOC Page Entries) ---')
    for c in corrections['toc_removed']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]}')

if corrections['added']:
    print('\n--- Added (False Negatives) ---')
    for c in corrections['added']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]} | matched: {c["matched"]} ({c["score"]} via {c["method"]})')

Correction Complete
  Kept (correct):        27
  Removed (false pos):   0
  Removed (TOC pages):   0
  Added (false neg):     1

--- Added (False Negatives) ---
  page  23 | INTERMEDIARY CERTIFICATE  | matched: Intermediary certificate (100.0 via ratio)


In [6]:
# ============================================================
# Save corrected JSON and correction log
# ============================================================

# Overwrite original file directly (avoid long path issue)
output_path = Path(JSON_PATH)
log_path = Path(JSON_PATH).parent / 'correction_log.json'

# Save corrected content_list (overwrite original)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(corrected, f, indent=4, ensure_ascii=False)

# Save correction log for auditability
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump({
        'source': str(JSON_PATH),
        'ground_truth': str(HITL_JSON_PATH),
        'stats': {
            'total_blocks': len(corrected),
            'kept': len(corrections['kept']),
            'removed_false_pos': len(corrections['removed']),
            'removed_toc': len(corrections['toc_removed']),
            'added_false_neg': len(corrections['added']),
        },
        'corrections': corrections,
    }, f, indent=2, ensure_ascii=False)

# Before vs After comparison
original_levels = sum(1 for b in content_list if 'text_level' in b)
corrected_levels = sum(1 for b in corrected if 'text_level' in b)

print(f'Saved corrected JSON:  {output_path}')
print(f'Saved correction log:  {log_path}')
print(f'\nBefore -> After:')
print(f'  text_level blocks: {original_levels} -> {corrected_levels}')
print(f'  Net change: {corrected_levels - original_levels:+d}')

Saved corrected JSON:  C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\g33a\auto\g33a_content_list copy.json
Saved correction log:  C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\g33a\auto\correction_log.json

Before -> After:
  text_level blocks: 27 -> 28
  Net change: +1


In [8]:
# ============================================================
# Validation Spot-Check
# ============================================================

# Show all text_level blocks in the corrected output
final_levels = [b for b in corrected if 'text_level' in b]

print(f'Final text_level Blocks ({len(final_levels)} total)')
print('=' * 70)
for i, b in enumerate(final_levels, 1):
    page = b.get('page_idx', '?')
    text = b.get('text', '')[:70]
    print(f'  {i:3d}. page {page:3d} | {text}')
print('=' * 70)

# Cross-check: which ground truth sections were NOT matched in the document?
matched_titles = set()
for c in corrections['kept'] + corrections['added']:
    matched_titles.add(c.get('matched', ''))

unmatched = [s for s in ground_truth_sections if s['title'] not in matched_titles]

if unmatched:
    print(f'\nWARNING: Ground truth sections NOT found in content_list.json ({len(unmatched)}):')
    for s in unmatched:
        sid = s.get('section_id') or 'PART'
        print(f'  [{sid:>5}] {s["title"]}')
else:
    print(f'\nAll ground truth sections matched in content_list.json')

Final text_level Blocks (28 total)
    1. page   2 | 1. Introduction 
    2. page   3 | 2. Customer acceptance policy 
    3. page   3 | 3. Customer due diligence 
    4. page   6 | 4. Corporate customers 
    5. page   7 | 5. Trust and nominee accounts 
    6. page   8 | 6. Reliance on intermediaries for customer due diligence 
    7. page  10 | 7. Client accounts 
    8. page  10 | 8. Non-face-to-face customers 
    9. page  11 | 9. Wire transfer messages 
   10. page  12 | 10. Politically exposed persons 
   11. page  14 | 11. Correspondent banking 
   12. page  15 | 12. Existing accounts 
   13. page  15 | 13. On-going monitoring 
   14. page  16 | 14. Jurisdictions which do not or insufficiently apply the FATF Recomm
   15. page  18 | 15. Terrorist financing 
   16. page  20 | 16. Risk management 
   17. page  22 | INTERMEDIARY CERTIFICATE 
   18. page  24 | INTERPRETATIVE NOTES 
   19. page  26 | Customer due diligence 
   20. page  29 | Corporate customers 
   21. page  33 | Tru